# Chapitre 1.4 : le transformer pièce par pièce

Le cœur de « ton IA » est un transformer. Dans ce carnet, tu le refais pièce par pièce, avec des fonctions toutes simples (pas de `nn.Module`, pas d’attention toute faite), puis tu vérifies qu’il donne les mêmes nombres que celui de forge, le moteur de « ton IA ». Tu travailles sur un tout petit modèle tiré au hasard : 2 couches, des vecteurs de 32 nombres, 4 têtes d’attention.

Ce carnet accompagne [la page du chapitre 1.4](https://chipat-neko.github.io/forge-ia/partie-1/1-4/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. donner un vecteur à chaque jeton (le plongement) ;
2. faire regarder une tête d’attention ;
3. lui interdire de regarder le futur (le masque) ;
4. faire regarder plusieurs têtes à la fois ;
5. dire à chaque jeton où il est (RoPE) ;
6. retravailler chaque jeton (SwiGLU) ;
7. assembler un bloc ;
8. comparer le modèle entier à forge ;
9. compter les paramètres de ton IA ;
10. relever un défi : écrire la softmax à la main, puis vérifier que ce carnet donne les mêmes résultats que la page du site.

**Comment l’utiliser**

Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

Ce carnet n’a pas besoin de carte graphique, et il n’installe rien : Colab a déjà PyTorch. Tous les calculs se font sur le processeur, en `float64` (des nombres décimaux sur 64 bits), sauf deux endroits où forge calcule en `float32` ; le carnet les signale en chemin.

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, puis indique à Python où trouver le projet et forge. Elle construit aussi le petit modèle de forge : ses poids sont tirés au hasard (graine 0), avec un écart-type de 0,2 pour que chaque pièce ait quelque chose à calculer. Les pièces du projet (`pieces.py`) reçoivent exactement ces poids. La cellule dit enfin sur quoi ce carnet tourne.

Elle définit aussi `nombre`, qui écrit les nombres à la française, avec une espace fine entre les milliers et une virgule.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia

projet = depot / "projets" / "1.4-transformer"
if not projet.exists():
    raise FileNotFoundError(
        "Le dossier projets/1.4-transformer est introuvable. Si le dépôt forge-ia était déjà là, "
        "supprime-le (panneau des fichiers, à gauche), puis relance cette cellule\u202f; sinon, vérifie ta connexion."
    )

# Python cherche les modules dans les dossiers de sys.path : on y ajoute le projet et forge.
sys.path.insert(0, str(projet.resolve()))
sys.path.insert(0, str((depot / "projets" / "forge").resolve()))

import torch
from forge.modele import ConfigModele

from transformer_pas_a_pas import pieces
from transformer_pas_a_pas.comparer import ecarts, poids_de_forge, preparer_forge

torch.set_default_device("cpu")
torch.manual_seed(0)

CONFIG = ConfigModele(dim=32, couches=2, tetes=4, longueur_contexte=16, taille_vocabulaire=50)
modele_forge = preparer_forge(CONFIG, graine=0)  # le vrai Transformer de forge, en float64
p = poids_de_forge(modele_forge)  # ses poids, rangés pour les fonctions de pieces.py
bloc0 = p["blocs"][0]
DIM_TETE = CONFIG.dim // CONFIG.tetes
BASE = CONFIG.base_rope


def nombre(x, decimales=0):
    # Un nombre à la française : « :, » sépare les milliers par des virgules, à l’anglaise ;
    # on les remplace par l’espace fine (\u202f), puis le point par une virgule.
    texte = f"{x + 0.0:,.{decimales}f}"
    return texte.replace(",", "\u202f").replace(".", ",")


def sci(x):
    # Un écart en notation scientifique à la française\u202f: « 8,9e-15 », ou « 0 » s’il est nul.
    return "0" if x == 0 else f"{x:.1e}".replace(".", ",")


print(f"Ce carnet calcule sur le processeur, avec PyTorch {torch.__version__.split('+')[0]}, en float64.")
print(f"Le petit modèle\u202f: {CONFIG.couches} couches, vecteurs de {CONFIG.dim} nombres, {CONFIG.tetes} têtes de {DIM_TETE} nombres, "
      f"{CONFIG.taille_vocabulaire} jetons possibles.")

## Étape 1 : le plongement

Un jeton est un nombre entier (son numéro dans le vocabulaire). Le modèle ne sait rien faire d’un numéro : la première pièce le remplace par un **vecteur**, une ligne d’une grande table (`plongement`). Cette table est apprise : c’est la plus grosse part des paramètres. La fonction ne fait que lire la ligne.

La cellule suivante plonge une suite de six jetons. Le jeton numéro 3 apparaît deux fois.

In [ ]:
ids = torch.tensor([7, 3, 41, 3, 12, 8])
x = pieces.plongement(ids, p["plongement"])

print(f"{len(ids)} jetons deviennent une table de {tuple(x.shape)[0]} lignes de {tuple(x.shape)[1]} nombres.")
print("Le jeton 3, en position 1\u202f:", [nombre(v, 2) for v in x[1, :4].tolist()], "…")
print("Le jeton 3, en position 3\u202f:", [nombre(v, 2) for v in x[3, :4].tolist()], "…")
print("Les deux vecteurs sont identiques\u202f:", bool(torch.equal(x[1], x[3])))

Le plongement donne le **même** vecteur au même jeton, où qu’il soit dans le texte. Le modèle ne sait donc pas encore dans quel ordre sont les jetons : l’étape 5 (RoPE) règle cela.

## Étape 2 : une tête d’attention

L’attention laisse chaque jeton regarder les autres. Chaque jeton produit trois vecteurs : une **requête** `q` (ce qu’il cherche), une **clé** `k` (ce qu’il propose) et une **valeur** `v` (ce qu’il transmet). Il compare sa requête à toutes les clés (un produit scalaire, divisé par √d pour garder des nombres raisonnables), ce qui donne des **scores**. La **softmax** transforme les scores d’un jeton en poids positifs qui totalisent 1. Le jeton reçoit enfin la moyenne des valeurs, pondérée par ces poids.

Avant l’attention, chaque vecteur passe par une RMSNorm, qui le ramène à une échelle raisonnable. La cellule suivante suit **une seule** tête (les 8 premiers nombres de `q`, `k` et `v`), à la main. RoPE n’est pas encore là (étape 5) : la tête ne sait pas encore où sont les jetons.

In [ ]:
h = pieces.rmsnorm(x, bloc0["norme_attention"])      # la normalisation avant l’attention
q, k, v = (h @ bloc0["w_qkv"].T).chunk(3, dim=-1)    # requêtes, clés, valeurs
q0, k0, v0 = q[:, :DIM_TETE], k[:, :DIM_TETE], v[:, :DIM_TETE]   # la première tête

scores = q0 @ k0.T / DIM_TETE ** 0.5                 # qui ressemble à qui
poids = pieces.softmax(scores)                       # chaque ligne totalise 1
sortie_tete = poids @ v0                             # la moyenne pondérée des valeurs

print("Les poids de la première tête (ligne = qui regarde, colonne = qui est regardé)\u202f:")
for ligne in poids:
    print("  " + "  ".join(nombre(w, 2) for w in ligne.tolist()))
print(f"Le jeton 0 donne {nombre(poids[0, 5].item(), 2)} de son attention au jeton 5, qui vient après lui\u202f: ce n’est pas permis.")

## Étape 3 : le masque

Pour **prédire** le jeton suivant, le modèle n’a pas le droit de regarder les jetons qui suivent : ce serait copier la réponse. Le **masque causal** met les scores de ces jetons à `-inf` (moins l’infini) avant la softmax ; comme `exp(-inf)` vaut 0, leur poids devient exactement 0.

In [ ]:
interdit = pieces.masque(len(ids))                   # True au-dessus de la diagonale
poids_masques = pieces.softmax(scores.masked_fill(interdit, float("-inf")))

print("Les mêmes poids, avec le masque\u202f:")
for ligne in poids_masques:
    print("  " + "  ".join(nombre(w, 2) for w in ligne.tolist()))
print("Chaque ligne totalise toujours 1\u202f:", bool(torch.allclose(poids_masques.sum(dim=-1), torch.ones(len(ids), dtype=torch.float64))))
print("Le jeton 0 ne regarde que lui-même\u202f:", poids_masques[0, 0].item() == 1.0)

# Preuve par le modèle entier\u202f: changer le DERNIER jeton ne change aucune sortie d’avant.
autres_ids = ids.clone()
autres_ids[-1] = 40
avant, apres = pieces.modele(ids, p), pieces.modele(autres_ids, p)
print("Les sorties des 5 premières positions ne changent pas\u202f:", bool(torch.equal(avant[:-1], apres[:-1])))

## Étape 4 : plusieurs têtes

Une seule tête ne suit qu’une relation à la fois. Le transformer en lance plusieurs en parallèle, chacune sur sa **tranche** des vecteurs (ici 4 têtes de 8 nombres). Chaque tête a sa propre façon de regarder. Leurs sorties sont recollées côte à côte, puis une dernière matrice (`w_sortie`) les mélange.

`poids_attention` rend les poids de toutes les têtes d’un coup : un tableau 4 × 6 × 6. `attention` fait tout le calcul, RoPE comprise (étape 5).

In [ ]:
tous_les_poids = pieces.poids_attention(h, bloc0["w_qkv"], CONFIG.tetes, BASE)
print("Forme des poids\u202f:", tuple(tous_les_poids.shape), "(têtes, jeton qui regarde, jeton regardé)")
for numero, tete in enumerate(tous_les_poids):
    print(f"  tête {numero}\u202f: le dernier jeton regarde surtout le jeton {tete[-1].argmax().item()} (poids {nombre(tete[-1].max().item(), 2)})")

sortie = pieces.attention(h, bloc0["w_qkv"], bloc0["w_sortie"], CONFIG.tetes, BASE)
rotation = pieces.angles_rope(len(ids), DIM_TETE, BASE)
cos, sin = rotation.cos(), rotation.sin()
attendue = modele_forge.blocs[0].attention(h.unsqueeze(0), cos, sin)[0]   # le module d’attention de forge
print(f"Écart avec l’attention de forge\u202f: {sci((sortie - attendue).abs().max().item())}")

## Étape 5 : RoPE

Jusqu’ici, mélanger l’ordre des jetons ne changerait rien aux poids (à part le masque). **RoPE** (« rotary position embedding ») donne la position **sans aucun paramètre** : chaque tête est découpée en paires de nombres, vues comme un point du plan, et chaque paire est **tournée** d’un angle proportionnel à la position du jeton. La paire `i` tourne de `position × base^(-2i / dim_tete)` radians : les premières paires tournent vite, les dernières lentement, comme les aiguilles d’une horloge.

Conséquence : le produit scalaire entre une requête à la position `m` et une clé à la position `n` ne dépend que de l’écart `m - n`.

**Où forge passe en `float32`.** Les angles, leurs cosinus et sinus, et la rotation sont calculés en `float32`, même dans un modèle en `float64` (et le résultat reprend ensuite le type de l’entrée). Pour retrouver forge à 10⁻¹² près, `pieces.py` fait pareil, aux mêmes endroits. C’est pourquoi le produit scalaire ci-dessous n’est égal qu’à environ 10⁻⁶ près.

In [ ]:
angles = pieces.angles_rope(6, DIM_TETE, BASE)
print("Les angles (radians), une ligne par position, une colonne par paire\u202f:")
for position, ligne in enumerate(angles.tolist()):
    print(f"  position {position}\u202f: " + "  ".join(nombre(a, 4).rjust(8) for a in ligne))
print("Type des angles\u202f:", angles.dtype, "(comme dans forge)")

# Deux vecteurs, placés à deux positions, puis décalés ensemble de 4 positions.
gen = torch.Generator().manual_seed(1)
requete = torch.randn(DIM_TETE, dtype=torch.float64, generator=gen)
cle = torch.randn(DIM_TETE, dtype=torch.float64, generator=gen)
grands_angles = pieces.angles_rope(12, DIM_TETE, BASE)


def produit(m, n):
    # La requête est tournée comme à la position m, la clé comme à la position n.
    qm = pieces.tourner(requete.expand(12, DIM_TETE), grands_angles)[m]
    kn = pieces.tourner(cle.expand(12, DIM_TETE), grands_angles)[n]
    return (qm @ kn).item()


print(f"Écart de 2 positions, en 5 et 3\u202f: {nombre(produit(5, 3), 4)}")
print(f"Écart de 2 positions, en 9 et 7\u202f: {nombre(produit(9, 7), 4)}")
print(f"Écart de 2 positions, en 2 et 0\u202f: {nombre(produit(2, 0), 4)}")
print(f"Écart de 5 positions, en 6 et 1\u202f: {nombre(produit(6, 1), 4)}\u202f: un autre écart, un autre produit.")

## Étape 6 : SwiGLU

Après l’attention, le MLP retravaille chaque jeton **séparément**. Le SwiGLU (la version de Llama et de ton IA) ajoute une **porte** : deux projections partent de `x`. L’une (la porte) passe par la fonction SiLU, `x / (1 + exp(-x))`, une courbe douce qui laisse passer les valeurs positives et étouffe les négatives. Les deux sont multipliées terme à terme, et une dernière projection ramène à la largeur du modèle.

In [ ]:
y = pieces.rmsnorm(x, bloc0["norme_mlp"])
sortie_mlp = pieces.swiglu(y, bloc0["w_porte"], bloc0["w_haute"], bloc0["w_basse"])

porte = y @ bloc0["w_porte"].T
silu = porte / (1 + torch.exp(-porte))
print(f"La porte a {porte.shape[-1]} composantes\u202f; avant SiLU, {nombre(100 * (porte < 0).double().mean().item())}\u202f% sont négatives.")
grille = torch.linspace(-10, 10, 200001, dtype=torch.float64)
plus_bas = (grille / (1 + torch.exp(-grille))).min().item()
print(f"Après SiLU, la plus négative vaut {nombre(silu.min().item(), 2)}\u202f: SiLU ne descend jamais sous {nombre(plus_bas, 4)} (son minimum, cherché sur une grille de -10 à 10).")
print(f"Écart avec le MLP de forge\u202f: {sci((sortie_mlp - modele_forge.blocs[0].mlp(y.unsqueeze(0))[0]).abs().max().item())}")

## Étape 7 : un bloc

Un bloc enchaîne l’attention puis le MLP, chacun **précédé** d’une RMSNorm et **ajouté** au vecteur (connexion résiduelle) : `x = x + attention(norme(x))`, puis `x = x + mlp(norme(x))`. Chaque sous-couche n’écrit qu’une correction, et l’information d’origine traverse tout le modèle.

In [ ]:
sortie_bloc = pieces.bloc(x, bloc0)
attendue_bloc = modele_forge.blocs[0](x.unsqueeze(0), cos, sin)[0]   # le Bloc de forge
print(f"Entrée du bloc\u202f: {tuple(x.shape)}, sortie\u202f: {tuple(sortie_bloc.shape)} (même forme, c’est ce qui permet de les empiler).")
print(f"La taille moyenne d’un vecteur passe de {nombre(x.norm(dim=-1).mean().item(), 2)} à {nombre(sortie_bloc.norm(dim=-1).mean().item(), 2)}\u202f: le bloc a ajouté ses corrections.")
print(f"Écart avec le bloc de forge\u202f: {sci((sortie_bloc - attendue_bloc).abs().max().item())}")

## Étape 8 : le modèle entier, comparé à forge

Le modèle entier : le plongement, les blocs l’un après l’autre, une dernière RMSNorm, puis une projection vers le vocabulaire qui donne un score (un **logit**) par jeton possible. Le plongement d’entrée et cette projection sont la **même** matrice (poids liés).

La fonction `ecarts` du projet compare chaque pièce au module correspondant de forge sur un même lot de deux textes, et donne le plus grand écart de chacune.

In [ ]:
logits = pieces.modele(ids, p)
attendus_logits = modele_forge(ids.unsqueeze(0))[0][0]
print(f"Logits\u202f: {tuple(logits.shape)} (un score pour chacun des {CONFIG.taille_vocabulaire} jetons, à chaque position).")
print(f"Le jeton le mieux noté après le dernier jeton\u202f: {logits[-1].argmax().item()} (forge\u202f: {attendus_logits[-1].argmax().item()}).")

resultat = ecarts(CONFIG)
noms = {"rmsnorm": "RMSNorm", "rope": "RoPE", "attention": "attention", "swiglu": "SwiGLU", "bloc": "un bloc", "modele": "le modèle entier"}
print("Plus grand écart avec forge, pièce par pièce\u202f:")
for cle_piece, nom in noms.items():
    print(f"  {nom:17}\u202f: {sci(resultat[cle_piece])}")
print(f"Identique à forge à 10⁻¹² près\u202f: {'oui' if max(resultat.values()) <= 1e-12 else 'non'}")

## Étape 9 : compter les paramètres de ton IA

Les paramètres sont les nombres que le modèle apprend : les matrices des étapes précédentes, et les poids des RMSNorm. RoPE n’en a aucun, et il n’y a aucun biais. Le projet les range en quatre familles : le **plongement** (compté une seule fois, puisque la sortie est la même matrice), l’**attention**, le **MLP** et les **normes**. La somme est le nombre exact de paramètres de forge, pour chacune des cinq tailles prévues.

In [ ]:
from forge.modele.config import PRESETS, nombre_parametres

from transformer_pas_a_pas.compter import repartition

print(f"{'taille':11} {'plongement':>16} {'attention':>16} {'MLP':>16} {'normes':>12} {'total':>16}")
somme_juste = True
for nom_preset, config in PRESETS.items():
    r = repartition(config)
    total = sum(r.values())
    somme_juste = somme_juste and total == nombre_parametres(config)
    print(f"{nom_preset:11} {nombre(r['plongement']):>16} {nombre(r['attention']):>16} {nombre(r['mlp']):>16} {nombre(r['normes']):>12} {nombre(total):>16}")
print("La somme est celle de forge pour les cinq tailles\u202f:", somme_juste)

## Exercice : écrire la softmax à la main

La softmax transforme des scores en poids positifs qui totalisent 1. On l’écrit en trois gestes : **soustraire le plus grand score** (sans quoi `exp` déborde sur les grands nombres ; le résultat ne change pas), prendre l’exponentielle de chaque score, puis **diviser** chacune par la somme des exponentielles. Elle se calcule sur le dernier axe (`dim=-1`) et doit accepter `-inf`, que le masque y met.

Écris la fonction `ma_softmax(x)` sans utiliser `torch.softmax`. La cellule l’essaie sur trois tenseurs (dont un avec de grands nombres et un avec `-inf`) et la compare à `torch.softmax`. Si l’écart reste sous 10⁻¹², le défi est réussi.

Indice : `x.max(dim=-1, keepdim=True).values` donne le plus grand score de chaque ligne, `torch.exp` l’exponentielle, et `.sum(dim=-1, keepdim=True)` la somme.

In [ ]:
def ma_softmax(x):
    # À toi : trois gestes (soustraire le plus grand score, exp, diviser par la somme), sur le dernier axe.
    return x


# Ne change rien en dessous : cette partie essaie ta fonction sur trois tenseurs.
essais = {
    "des scores ordinaires": torch.randn(4, 7, dtype=torch.float64, generator=torch.Generator().manual_seed(2)),
    "de grands scores": torch.tensor([[1000.0, 999.0, 998.0]], dtype=torch.float64),
    "un score à -inf": torch.tensor([[0.5, float("-inf"), 2.0], [float("-inf"), float("-inf"), 1.0]], dtype=torch.float64),
}
reussis = 0
for description, essai in essais.items():
    voulu = torch.softmax(essai, dim=-1)
    obtenu = ma_softmax(essai)
    if torch.is_tensor(obtenu) and obtenu.shape == voulu.shape and torch.isfinite(obtenu).all() and (obtenu - voulu).abs().max() <= 1e-12:
        reussis += 1
    else:
        print(f"Avec {description}, ta fonction ne donne pas ce que donne torch.softmax.")
if reussis == len(essais):
    print("Défi réussi\u202f: ta softmax égale celle de PyTorch à 10⁻¹² près.")
else:
    print(f"Pas encore ({reussis} essai sur {len(essais)}).")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

On soustrait d’abord le plus grand score de chaque ligne : le plus grand devient 0 (son `exp` vaut 1), les autres sont négatifs, et `exp` ne peut plus déborder. Un score à `-inf` reste à `-inf`, et son `exp` vaut 0. Puis on divise par la somme.

Remplace la fonction de la cellule de l’exercice par :

```python
def ma_softmax(x):
    decale = x - x.max(dim=-1, keepdim=True).values
    exponentielles = torch.exp(decale)
    return exponentielles / exponentielles.sum(dim=-1, keepdim=True)
```

</details>

## Dernière étape : les mêmes résultats que la page ?

La page du site n’utilise pas Python : elle refait deux petits calculs en TypeScript, directement dans ton navigateur : la répartition des paramètres entre les familles, et les angles de RoPE de la figure. Pour être sûr que les deux donnent les mêmes résultats, le dépôt garde un fichier de référence, `tests/attendus.json`, calculé par le Python, que les tests de la page relisent. La cellule suivante recalcule tout avec les fonctions du projet et le compare : les répartitions (des entiers) exactement, les angles (qui passent par `pow`, `cos` et `sin`, un peu différents d’un système à l’autre) à 10⁻⁹ près.

Les angles de la figure restent en `float64`, alors que ceux de forge sont en `float32` : la figure montre le calcul, forge fait celui du modèle.

In [ ]:
import json
import math

from transformer_pas_a_pas.attendus import angles_de_la_figure

attendus = json.loads((projet / "tests" / "attendus.json").read_text(encoding="utf-8"))

# La répartition des paramètres de chaque taille, comparée exactement.
repartitions_justes = all(
    {cle: ligne[cle] for cle in ("plongement", "attention", "mlp", "normes")} == repartition(PRESETS[ligne["preset"]])
    for ligne in attendus["repartitions"]
)
print(f"{len(attendus['repartitions'])} tailles de forge comparées, {len(attendus['rope']['angles'])} positions de RoPE comparées.")
print(f"Même répartition des paramètres que la page du site\u202f: {'oui' if repartitions_justes else 'non'}")

# Les angles de la figure (4 paires d’une tête de 8, positions 0 à 32), comparés à 10⁻⁹ près.
mes_angles = angles_de_la_figure()
angles_justes = all(
    math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-12)
    for ligne_a, ligne_b in zip(mes_angles, attendus["rope"]["angles"])
    for a, b in zip(ligne_a, ligne_b)
)
print(f"Même tableau d’angles de RoPE que la page du site\u202f: {'oui' if angles_justes else 'non'}")
print(f"Ce carnet a tourné sur le processeur, avec PyTorch {torch.__version__.split('+')[0]}.")

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/1.4-transformer sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/1.4-transformer).
- La page du chapitre, pour voir le transformer s’assembler et lire le code expliqué : [chapitre 1.4](https://chipat-neko.github.io/forge-ia/partie-1/1-4/).
- Depuis le dossier du projet, `uv run --extra cpu python -m transformer_pas_a_pas comparer` refait la comparaison pièce par pièce, et `uv run --extra cpu python -m transformer_pas_a_pas compter --preset tres_grand` compte les paramètres d’une taille de forge.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.